# CellWhisperer Zero-shot Prediction Benchmark

Reimplements the zero-shot prediction logic from plot_confusion_matrix_and_get_performance_metrics for trained SpotWhisperer models.

In [ ]:
import warnings
# avoid DeprecationWarning
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pandas.core.algorithms")

In [ ]:
import pandas as pd
import logging
import numpy as np
import copy
import matplotlib
import torch
import seaborn as sns
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
from cellwhisperer.validation.zero_shot.single_cell_annotation import (
    get_performance_metrics_transcriptome_vs_text
)
from cellwhisperer.utils.model_io import load_cellwhisperer_model
from zero_shot_validation_scripts.utils import TABSAP_WELLSTUDIED_COLORMAPPING, PANCREAS_ORDER, SUFFIX_PREFIX_DICT
from zero_shot_validation_scripts.dataset_preparation import load_and_preprocess_dataset

sc.set_figure_params(
    vector_friendly=True, dpi_save=500
)

In [ ]:
#### Parameters ####

matplotlib.style.use(snakemake.input.mpl_style)

dataset_name = snakemake.wildcards.dataset
metadata_col = snakemake.wildcards.metadata_col
model_name = snakemake.wildcards.model_name

print(f"Running CellWhisperer benchmark for:")
print(f"  Dataset: {dataset_name}")
print(f"  Metadata column: {metadata_col}")
print(f"  Model: {model_name}")

In [ ]:
# Load model
(
    pl_model_cellwhisperer,
    text_processor_cellwhisperer,
    cellwhisperer_transcriptome_processor,
) = load_cellwhisperer_model(model_path=snakemake.input.model, eval=True)
cellwhisperer_model = pl_model_cellwhisperer.model

print(f"Model loaded successfully: {type(cellwhisperer_model)}")

In [ ]:
# Load data
adata = load_and_preprocess_dataset(
    dataset_name=dataset_name, 
    read_count_table_path=snakemake.input.raw_read_count_table,
    obsm_paths={
        "X_cellwhisperer": (snakemake.input.processed_dataset, "transcriptome_embeds"),
        "X_geneformer": (snakemake.input.processed_dataset, "transcriptome_features")
    }
)
logging.info(f"Data loaded and preprocessed. Shape: {adata.shape}")

In [ ]:
# Run zero-shot prediction using the same logic as the original implementation
confusion_matrix, performance_metrics, performance_metrics_per_metadata = get_performance_metrics_transcriptome_vs_text(
    adata=adata,
    metadata_col=metadata_col,
    cellwhisperer_model=cellwhisperer_model,
    text_processor=text_processor_cellwhisperer,
    use_prefix_suffix_version=snakemake.params.use_prefix_suffix_version,
    suffix_prefix_dict=SUFFIX_PREFIX_DICT
)

print(f"Performance metrics computed:")
print(f"  Accuracy: {performance_metrics.get('accuracy', 'N/A')}")
print(f"  F1 score: {performance_metrics.get('f1_weighted', 'N/A')}")

In [ ]:
# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    confusion_matrix,
    cmap="Blues",
    annot=False,
    square=True,
    cbar_kws={"shrink": 0.7},
    vmin=0,
    vmax=1 if snakemake.params.normed else None,
)
plt.yticks(
    [x + 0.5 for x in range(len(confusion_matrix.index))],
    confusion_matrix.index,
)
plt.xticks(
    [x + 0.5 for x in range(len(confusion_matrix.columns))],
    confusion_matrix.columns,
    rotation=45,
    ha="right",
)
plt.xlabel("Best-matching keyword")
plt.ylabel("True class")
plt.title(f"Confusion Matrix - {dataset_name} ({model_name})")
plt.tight_layout()

# Save confusion matrix plot
plt.savefig(snakemake.output.confusion_matrix_plot, dpi=300, bbox_inches='tight')
plt.close()

print(f"Confusion matrix plot saved to: {snakemake.output.confusion_matrix_plot}")

In [ ]:
# Save results

# Ensure output directories exist
for output_path in [snakemake.output.confusion_matrix_table, 
                   snakemake.output.performance_metrics,
                   snakemake.output.performance_metrics_per_metadata]:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

# Save confusion matrix as xlsx
confusion_matrix.to_excel(snakemake.output.confusion_matrix_table, index=True)

# Save performance metrics
pd.DataFrame([performance_metrics]).to_csv(
    snakemake.output.performance_metrics, index=False
)

# Save per-metadata performance metrics
performance_metrics_per_metadata.to_csv(
    snakemake.output.performance_metrics_per_metadata, index=True
)

print(f"Results saved:")
print(f"  Confusion matrix: {snakemake.output.confusion_matrix_table}")
print(f"  Performance metrics: {snakemake.output.performance_metrics}")
print(f"  Per-metadata metrics: {snakemake.output.performance_metrics_per_metadata}")